In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import requests

In [ ]:
url="https://raw.githubusercontent.com/GeethaGunasekaran1/Dataset_rep/main/social_media_engagement_5000.csv"

In [ ]:
import pandas as pd
df = pd.read_csv(url)

In [ ]:

df = pd.read_csv(url)

print("DataFrame reloaded successfully. Displaying the first 5 rows to confirm:")
display(df.head())

print("\nVerifying data types to ensure column names and types are restored:")
display(df.dtypes)

In [ ]:

df['posted_at'] = pd.to_datetime(df['posted_at'], format='%d-%m-%Y', errors='coerce')

print("Data type of 'posted_at' after conversion:")
display(df['posted_at'].dtype)


print("\nDataFrame head after 'posted_at' conversion:")
display(df[['posted_at']].head())

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

In [ ]:
df["posted_at"] = pd.to_datetime(
    df["posted_at"],
    format="%d-%m-%Y",
    errors="coerce"
)


In [ ]:
df.dtypes

In [ ]:
numeric_columns = [
    "age",
    "likes",
    "comments",
    "shares",
    "watch_time_sec",
    "impression_count",
    "follower_count",
    "engagement_rate"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)

In [ ]:
likes_array = df["likes"].dropna().to_numpy()
print(likes_array[:10])

In [ ]:
print(np.sum(likes_array))


In [ ]:
print(np.nanmedian(likes_array))

In [ ]:
print(np.nanmean(likes_array))

In [ ]:
print(np.nanmax(likes_array))

In [ ]:
print(np.nanmin(likes_array))

# Data Cleaning

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
numeric_columns = [
    "age",
    "likes",
    "comments",
    "shares",
    "watch_time_sec",
    "impression_count",
    "follower_count",
    "engagement_rate"
]

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
categorical_columns = [
    "gender",
    "country",
    "post_type",
    "post_category",
    "is_verified",
    "device_type",
    "sentiment",
    "hashtags"
]

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
text_columns = [
    "gender",
    "country",
    "post_type",
    "post_category",
    "device_type",
    "sentiment"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

In [ ]:
df["gender"] = df["gender"].str.title()

df["gender"] = df["gender"].replace({
    "Male": "Male",
    "Female": "Female",
    "Other": "Other"
})

In [ ]:
engagement_columns = [
    'likes', 'comments', 'shares', 'impressions',
    'watch_time', 'followers', 'engagement_rate'
]
for col in engagement_columns:
    if col in df.columns:
        df.loc[df[col] < 0, col] = np.nan

In [ ]:
df["sentiment"] = df["sentiment"].str.lower().str.strip()

df["sentiment"] = df["sentiment"].replace({
    "pos": "positive",
    "neg": "negative",
    "neu": "neutral"
})

In [ ]:
print("Negative likes:", (df["likes"] < 0).sum())

In [ ]:
print("Negative comments:", (df["comments"] < 0).sum())


In [ ]:
print("Negative shares:", (df["shares"] < 0).sum())
print("Negative watch time:", (df["watch_time_sec"] < 0).sum())
print("Negative impressions:", (df["impression_count"] < 0).sum())
print("Negative followers:", (df["follower_count"] < 0).sum())

In [ ]:
count_columns = [
    "likes",
    "comments",
    "shares",
    "watch_time_sec",
    "impression_count",
    "follower_count"
]

for col in count_columns:
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df[col].fillna(df[col].median())

In [ ]:


invalid_rate = df["engagement_rate"] > 1

print("Invalid engagement rates:", invalid_rate.sum())

valid_median = df.loc[
    df["engagement_rate"] <= 1,
    "engagement_rate"
].median()

df.loc[
    df["engagement_rate"] > 1,
    "engagement_rate"
] = valid_median

print("Invalid values after cleaning:",
      (df["engagement_rate"] > 1).sum())

In [ ]:
df["hashtag_count"] = (
    df["hashtags"]
    .astype(str)
    .str.findall(r"#\w+")
    .str.len()
)

df[["hashtags", "hashtag_count"]].head(10)

In [ ]:
df.describe()

In [ ]:
print(df["gender"].unique())
print(df["gender"].nunique())
print(df["gender"].value_counts())

In [ ]:
print(df["country"].unique())
print(df["country"].nunique())
print(df["country"].value_counts())

In [ ]:
print(df["post_type"].unique())
print(df["post_type"].value_counts())

In [ ]:
print(df["post_category"].unique())
print(df["post_category"].value_counts())

In [ ]:
print(df["sentiment"].unique())
print(df["sentiment"].value_counts())

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

correlation_matrix = numeric_df.corr()

correlation_matrix

In [ ]:
df["engagement_score"] = (
    df["likes"] +
    (2 * df["comments"]) +
    (3 * df["shares"])
) / df["impression_count"].replace(0, np.nan)

df["engagement_score"] = df["engagement_score"].fillna(0)

df[[
    "likes",
    "comments",
    "shares",
    "impression_count",
    "engagement_score"
]].head()

In [ ]:
df["log_likes"] = np.log1p(df["likes"])

df[["likes", "log_likes"]].head()

In [ ]:
post_type_summary = df.groupby("post_type").agg(
    average_likes=("likes", "mean"),
    average_comments=("comments", "mean"),
    average_shares=("shares", "mean"),
    average_engagement_rate=("engagement_rate", "mean"),
    total_posts=("post_id", "count")
).sort_values(
    "average_engagement_rate",
    ascending=False
)

post_type_summary

In [ ]:
country_summary = df.groupby("country").agg(
    average_likes=("likes", "mean"),
    average_comments=("comments", "mean"),
    average_shares=("shares", "mean"),
    average_engagement_rate=("engagement_rate", "mean"),
    total_posts=("post_id", "count")
).sort_values(
    "average_engagement_rate",
    ascending=False
)

country_summary

In [ ]:
category_summary = df.groupby("post_category").agg(
    average_likes=("likes", "mean"),
    average_comments=("comments", "mean"),
    average_shares=("shares", "mean"),
    average_engagement_rate=("engagement_rate", "mean"),
    total_posts=("post_id", "count")
).sort_values(
    "average_engagement_rate",
    ascending=False
)

category_summary

In [ ]:
sentiment_summary = df.groupby("sentiment").agg(
    average_likes=("likes", "mean"),
    average_comments=("comments", "mean"),
    average_shares=("shares", "mean"),
    average_engagement_rate=("engagement_rate", "mean"),
    average_followers=("follower_count", "mean"),
    total_posts=("post_id", "count")
).sort_values(
    "average_engagement_rate",
    ascending=False
)

sentiment_summary

In [ ]:
stats_columns = [
    "likes",
    "comments",
    "shares",
    "watch_time_sec",
    "engagement_rate",
    "follower_count"
]

statistics = pd.DataFrame(index=stats_columns)

statistics["Mean"] = df[stats_columns].mean()
statistics["Median"] = df[stats_columns].median()
statistics["Mode"] = df[stats_columns].mode().iloc[0]

statistics

In [ ]:
statistics["Standard Deviation"] = df[stats_columns].std()
statistics["Variance"] = df[stats_columns].var()

statistics

In [ ]:
percentiles = df[stats_columns].quantile(
    [0.25, 0.50, 0.75]
)

percentiles

In [ ]:
statistics["Skewness"] = df[stats_columns].skew()
statistics["Kurtosis"] = df[stats_columns].kurt()

statistics

# Matplotlib Visualizations

# **Scatter — Likes vs Impressions**

In [ ]:


plt.scatter(
    df["impression_count"],
    df["likes"]
)

plt.xlabel("Impressions")
plt.ylabel("Likes")
plt.title("Likes vs Impressions")
plt.figure(figsize=(8, 5))

plt.show()

# **Line — Daily Engagement Trend**

In [ ]:
daily_engagement = df.groupby('posted_at')['engagement_rate'].mean().reset_index()

plt.plot(
    daily_engagement["posted_at"],
    daily_engagement["engagement_rate"]
)

plt.xlabel("Date")
plt.ylabel("Average Engagement Rate")
plt.title("Daily Engagement Trend")

plt.xticks(rotation=45)
plt.tight_layout()
plt.figure(figsize=(10,6))

plt.show()

# **Bar — Posts by Category**

In [ ]:
category_counts = df['post_category'].value_counts()

plt.bar(
    category_counts.index,
    category_counts.values
)

plt.xlabel("Post Category")
plt.ylabel("Number of Posts")
plt.title("Posts by Category")

plt.xticks(rotation=50)
plt.tight_layout()
plt.figure(figsize=(10,6))

plt.show()

# **Pie — Gender Distribution**

In [ ]:
gender_counts = df["gender"].value_counts()


plt.pie(
    gender_counts.values,
    labels=gender_counts.index,

)

plt.title("Gender Distribution")
plt.figure(figsize=(7, 7))
plt.show()

# **Histogram — Age**

In [ ]:


plt.hist(
    df["age"],
    bins=15,

)

plt.xlabel("Age")
plt.ylabel("Frequency")
plt.title("Age Distribution")
plt.figure(figsize=(8, 5))
plt.show()

# **Box Plot — Engagement Rate**

In [ ]:


plt.boxplot(
    df["engagement_rate"],
    vert=True
)

plt.ylabel("Engagement Rate")
plt.title("Engagement Rate Distribution")
plt.figure(figsize=(8,5))

plt.show()

# **SEABORN**

# **Count Plot — Post Type**

In [ ]:


sns.countplot(
    data=df,
    x="post_type"
)

plt.title("Post Type Distribution")
plt.xlabel("Post Type")
plt.ylabel("Count")
plt.figure(figsize=(8,5))

plt.show()

# **Bar Plot — Average Likes by Category**

In [ ]:


sns.barplot(
    data=df,
    x="post_category",
    y="likes",
    estimator="mean"
)

plt.title("Average Likes by Category")
plt.xlabel("Post Category")
plt.ylabel("Average Likes")

plt.xticks(rotation=45)
plt.tight_layout()
plt.figure(figsize=(10, 5))

plt.show()

# **Violin Plot — Followers vs Sentiment**

In [ ]:


sns.violinplot(
    data=df,
    x="sentiment",
    y="follower_count"
)

plt.title("Follower Count vs Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Follower Count")
plt.figure(figsize=(8, 5))

plt.show()

# **Pair Plot — Numeric Features**

In [ ]:
pair_columns = [
    "likes",
    "comments",
    "shares",
    "engagement_rate",
    "follower_count"
]

sns.pairplot(
    df[pair_columns].sample(
        min(1000, len(df)),
        random_state=42
    )
)

plt.show()

# **Correlation Heatmap**

In [ ]:

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Correlation Matrix Heatmap")
plt.figure(figsize=(12, 8))


plt.show()

**Plotly (Interactive)**


# **Line Chart**

In [ ]:
fig = px.line(
    daily_engagement,
    x="posted_at",
    y="engagement_rate",
    title="Interactive Daily Engagement Trend",
    labels={
        "posted_at": "Date",
        "engagement_rate": "Average Engagement Rate"
    }
)

fig.show()

# **Bar Chart**

In [ ]:
country_plot = (
    df.groupby("country", as_index=False)
    ["engagement_rate"]
    .mean()
    .sort_values(
        "engagement_rate",
        ascending=False
    )
)

fig = px.bar(
    country_plot,
    x="country",
    y="engagement_rate",
    title="Average Engagement Rate by Country",
    labels={
        "country": "Country",
        "engagement_rate": "Average Engagement Rate"
    }
)

fig.show()

# **Bubble Chart**

In [ ]:
fig = px.scatter(
    df.sample(min(1000, len(df)), random_state=42),
    x="impression_count",
    y="likes",
    size="follower_count",
    color="post_type",
    hover_data=[
        "country",
        "post_category",
        "sentiment",
        "engagement_rate"
    ],
    title="Likes vs Impressions — Interactive Bubble Chart"
)

fig.show()

In [ ]:
post_type_result = (
    df.groupby("post_type")["engagement_rate"]
    .mean()
    .sort_values(ascending=False)
)

print(post_type_result)
print("\nHighest average engagement post type:")
print(post_type_result.index[0])

In [ ]:
category_result = (
    df.groupby("post_category")["engagement_rate"]
    .mean()
    .sort_values(ascending=False)
)

print(category_result)

print("\nHighest average engagement category:")
print(category_result.index[0])

In [ ]:
country_result = (
    df.groupby("country")["engagement_rate"]
    .mean()
    .sort_values(ascending=False)
)

print(country_result)

In [ ]:
age_correlation = df[
    ["age", "engagement_rate"]
].corr()

print(age_correlation)

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 17, 25, 35, 45, 55, 65, 100],
    labels=[
        "Under 18",
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56-65",
        "65+"
    ]
)

age_group_analysis = df.groupby(
    "age_group",
    observed=True
)["engagement_rate"].mean()

print(age_group_analysis)

In [ ]:
sentiment_analysis = df.groupby("sentiment").agg(
    average_likes=("likes", "mean"),
    average_comments=("comments", "mean"),
    average_shares=("shares", "mean"),
    average_engagement_rate=("engagement_rate", "mean"),
    number_of_posts=("post_id", "count")
)

sentiment_analysis

In [ ]:
negative_neutral = df[
    df["sentiment"].isin(["negative", "neutral"])
]

negative_neutral_summary = negative_neutral.groupby(
    "sentiment"
).agg(
    avg_likes=("likes", "mean"),
    avg_comments=("comments", "mean"),
    avg_shares=("shares", "mean"),
    avg_engagement_rate=("engagement_rate", "mean"),
    avg_followers=("follower_count", "mean")
)

negative_neutral_summary

In [ ]:
print(
    df["hashtag_count"].value_counts().sort_index()
)

In [ ]:
print(
    "Average hashtag count:",
    df["hashtag_count"].mean()
)

In [ ]:
device_analysis = df.groupby(
    "device_type"
).agg(
    average_engagement_rate=("engagement_rate", "mean"),
    average_likes=("likes", "mean"),
    number_of_posts=("post_id", "count")
).sort_values(
    "average_engagement_rate",
    ascending=False
)

device_analysis


In [ ]:
verification_analysis = df.groupby(
    "is_verified"
).agg(
    average_engagement_rate=("engagement_rate", "mean"),
    average_likes=("likes", "mean"),
    average_followers=("follower_count", "mean"),
    number_of_posts=("post_id", "count")
)

verification_analysis

In [ ]:
df["day_name"] = df["posted_at"].dt.day_name()

day_analysis = df.groupby(
    "day_name"
)["engagement_rate"].mean()

day_analysis = day_analysis.sort_values(
    ascending=False
)

day_analysis

In [ ]:
df["day_name"] = df["posted_at"].dt.day_name()

day_analysis = df.groupby(
    "day_name"
)["engagement_rate"].mean()

day_analysis = day_analysis.sort_values(
    ascending=False
)

day_analysis